# Unit Tests — Transform Logic

This notebook contains unit tests for the ETL transform logic using PySpark.

**Tests cover**:
- Deduplication of records
- Text column trimming and standardization
- Derived column calculations (`total_amount`)
- Data quality filtering (null `order_id`, negative quantities)
- Time-based enrichment columns (`order_month`, `order_year`)

**How to run**: Execute this notebook directly or as part of the CI/CD pipeline.

In [0]:
# Databricks notebook source
# --- Unit Tests for ETL Transform Logic ---

from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col, trim, upper, month, year, date_format
import unittest

# Create the test spark session
spark_test = spark

# Helper: create a sample dataframe for testing
def create_test_df(spark, data):
    """Create a test DataFrame from a list of dicts."""
    return spark.createDataFrame(data)

# Define the transform function (mirrors the transform notebook logic)
def transform_sales(df):
    """Apply the same transformation logic as the transform notebook."""
    from pyspark.sql.functions import col, trim, upper, month, year, date_format, when
    from pyspark.sql.types import DecimalType
    
    # Deduplicate
    deduped = df.distinct()
    
    # Clean text columns
    cleaned = (
        deduped
        .withColumn("customer_name", trim(col("customer_name")))
        .withColumn("product_name", trim(col("product_name")))
        .withColumn("category", trim(col("category")))
        .withColumn("region", upper(trim(col("region"))))
    )
    
    # Data quality filter
    valid = cleaned.filter(
        col("order_id").isNotNull()
        & (col("quantity") > 0)
        & (col("unit_price") > 0)
    )
    
    # Derived columns
    transformed = (
        valid
        .withColumn("total_amount", (col("quantity") * col("unit_price")).cast("decimal(10,2)"))
        .withColumn("order_month", month(col("order_date")))
        .withColumn("order_year", year(col("order_date")))
        .withColumn("order_date_str", date_format(col("order_date"), "yyyy-MM-dd"))
    )
    
    return transformed
print("Test utilities loaded.")

In [0]:
# Databricks notebook source
# --- Run all tests ---

class TestTransformSales(unittest.TestCase):
    
    @classmethod
    def setUpClass(cls):
        cls.spark = spark_test
        cls.sample_data = [
            Row(order_id=1, order_date="2024-01-15", customer_id="C001",
                customer_name="  John Smith  ", product_id="P001",
                product_name="Laptop", category="Electronics",
                quantity=2, unit_price=1200.00, region="north america"),
            Row(order_id=2, order_date="2024-01-16", customer_id="C002",
                customer_name="Jane Doe", product_id="P002",
                product_name="Phone", category="Electronics",
                quantity=3, unit_price=800.00, region="europe"),
            # Duplicate of order_id=1
            Row(order_id=1, order_date="2024-01-15", customer_id="C001",
                customer_name="  John Smith  ", product_id="P001",
                product_name="Laptop", category="Electronics",
                quantity=2, unit_price=1200.00, region="north america"),
            # Invalid: null order_id
            Row(order_id=None, order_date="2024-01-17", customer_id="C003",
                customer_name="Bob", product_id="P003",
                product_name="Headphones", category="Electronics",
                quantity=1, unit_price=100.00, region="asia"),
            # Invalid: negative quantity
            Row(order_id=4, order_date="2024-01-18", customer_id="C004",
                customer_name="Alice", product_id="P004",
                product_name="Shoes", category="Apparel",
                quantity=-5, unit_price=50.00, region="europe"),
        ]
        cls.sample_df = create_test_df(cls.spark, cls.sample_data)
    
    def test_deduplication(self):
        """Duplicate rows should be removed."""
        result = transform_sales(self.sample_df)
        # 5 input rows, 1 exact duplicate, 2 invalid → 2 valid unique rows
        self.assertEqual(result.count(), 2, f"Expected 2 rows after dedup + quality filter, got {result.count()}")
    
    def test_whitespace_trimming(self):
        """Text columns should have whitespace trimmed."""
        result = transform_sales(self.sample_df)
        john = result.filter(col("order_id") == 1).collect()[0]
        self.assertEqual(john["customer_name"], "John Smith")
    
    def test_region_uppercase(self):
        """Region should be uppercased."""
        result = transform_sales(self.sample_df)
        john = result.filter(col("order_id") == 1).collect()[0]
        self.assertEqual(john["region"], "NORTH AMERICA")
    
    def test_total_amount_calculation(self):
        """total_amount should equal quantity * unit_price."""
        result = transform_sales(self.sample_df)
        john = result.filter(col("order_id") == 1).collect()[0]
        self.assertEqual(float(john["total_amount"]), 2400.00)
    
    def test_null_order_id_filtered(self):
        """Records with null order_id should be filtered out."""
        result = transform_sales(self.sample_df)
        null_orders = result.filter(col("order_id").isNull()).count()
        self.assertEqual(null_orders, 0, "No null order_id rows should remain")
    
    def test_negative_quantity_filtered(self):
        """Records with negative quantity should be filtered out."""
        result = transform_sales(self.sample_df)
        neg_qty = result.filter(col("quantity") <= 0).count()
        self.assertEqual(neg_qty, 0, "No negative quantity rows should remain")
    
    def test_order_month_extraction(self):
        """order_month should be correctly extracted from order_date."""
        result = transform_sales(self.sample_df)
        john = result.filter(col("order_id") == 1).collect()[0]
        self.assertEqual(john["order_month"], 1)
    
    def test_order_year_extraction(self):
        """order_year should be correctly extracted from order_date."""
        result = transform_sales(self.sample_df)
        john = result.filter(col("order_id") == 1).collect()[0]
        self.assertEqual(john["order_year"], 2024)
    
    def test_order_date_str_format(self):
        """order_date_str should be in yyyy-MM-dd format."""
        result = transform_sales(self.sample_df)
        john = result.filter(col("order_id") == 1).collect()[0]
        self.assertEqual(john["order_date_str"], "2024-01-15")

# Run tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestTransformSales)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

if not result.wasSuccessful():
    raise Exception(f"{result.failures.__len__() + result.errors.__len__()} test(s) failed")
else:
    print(f"\nAll {result.testsRun} tests passed!")